In [15]:
import sys
import os
from tqdm import tqdm
# tambahin root project ke path
sys.path.append(os.path.abspath(".."))

In [2]:
from utils.dataloader import get_dataloader

In [3]:
BASE_DIR = os.path.abspath("..")

train_path = os.path.join(BASE_DIR, "data/WLBisindo/split/train")
val_path   = os.path.join(BASE_DIR, "data/WLBisindo/split/val")
test_path  = os.path.join(BASE_DIR, "data/WLBisindo/split/test")

print(train_path)

D:\Softwares\AnacondaProjects\GestureSequenceCNN\data/WLBisindo/split/train


In [4]:
import os

print("Train exists:", os.path.exists(train_path))
print("Val exists:", os.path.exists(val_path))
print("Test exists:", os.path.exists(test_path))

Train exists: True
Val exists: True
Test exists: True


In [5]:
train_loader = get_dataloader(train_path)
val_loader   = get_dataloader(val_path, shuffle=False)
test_loader  = get_dataloader(test_path, shuffle=False)

In [6]:
for x, y in train_loader:
    print("Input shape :", x.shape)  # (B, 20, 3, 224, 224)
    print("Label shape :", y.shape) # (B,)
    print("Label sample:", y[:5])
    break

Input shape : torch.Size([8, 20, 3, 224, 224])
Label shape : torch.Size([8])
Label sample: tensor([30,  4, 22,  8, 16])


In [7]:
import os
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter

# buat folder run unik
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

BASE_DIR = os.path.abspath("..")
log_dir = os.path.join(BASE_DIR, "outputs/logs/cnn_lstm", f"run_{timestamp}")

os.makedirs(log_dir, exist_ok=True)

writer = SummaryWriter(log_dir)

print("Log dir:", log_dir)

Log dir: D:\Softwares\AnacondaProjects\GestureSequenceCNN\outputs/logs/cnn_lstm\run_20260408_235004


In [8]:
log_file = os.path.join(log_dir, "log.txt")

def log_message(msg):
    print(msg)
    with open(log_file, "a") as f:
        f.write(msg + "\n")

In [9]:
# Tensor Board
# type in terminal = tensorboard --logdir=outputs/logs
# http://localhost:6006

In [10]:
# ambil jumlah kelas dari dataset
NUM_CLASSES = len(train_loader.dataset.label_map)

print("Num classes:", NUM_CLASSES)

Num classes: 32


In [11]:
from models.cnn_lstm import CNN_LSTM

model = CNN_LSTM(num_classes=NUM_CLASSES)

x, y = next(iter(train_loader))
out = model(x)

print(out.shape)

torch.Size([8, 32])


In [12]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = CNN_LSTM(num_classes=NUM_CLASSES).to(device)

for param in model.encoder.cnn.parameters():
    param.requires_grad = False

criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    patience=2,
    factor=0.5
)

Device: cuda


In [13]:
def compute_accuracy(preds, labels):
    preds = torch.argmax(preds, dim=1)
    correct = (preds == labels).sum().item()
    return correct / len(labels)

In [16]:
EPOCHS = 10
patience = 3
counter = 0
best_val_loss = float("inf")

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    # ===== TRAIN =====
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(train_loader, desc="Training", leave=False)

    for x, y in train_bar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        train_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        train_correct += (preds == y).sum().item()
        train_total += y.size(0)

        # 🔥 update progress bar
        train_bar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    train_loss /= len(train_loader)
    train_acc = train_correct / train_total


    # ===== VALIDATION =====
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    val_bar = tqdm(val_loader, desc="Validation", leave=False)

    with torch.no_grad():
        for x, y in val_bar:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            outputs = model(x)
            loss = criterion(outputs, y)

            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            val_correct += (preds == y).sum().item()
            val_total += y.size(0)

            # 🔥 update progress bar
            val_bar.set_postfix({
                "loss": f"{loss.item():.4f}"
            })

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total


    # ===== LOGGING =====
    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Loss/val", val_loss, epoch)

    writer.add_scalar("Accuracy/train", train_acc, epoch)
    writer.add_scalar("Accuracy/val", val_acc, epoch)

    current_lr = optimizer.param_groups[0]['lr']
    writer.add_scalar("LR", current_lr, epoch)

    log_message(f"Epoch {epoch+1}/{EPOCHS}")
    log_message(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
    log_message(f"Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f}")
    log_message(f"LR: {current_lr}")
    log_message("-" * 40)


    # ===== SCHEDULER =====
    scheduler.step(val_loss)


    # ===== EARLY STOPPING =====
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0

        torch.save(model.state_dict(), os.path.join(log_dir, "best_model.pth"))
        log_message("Best model saved!")

    else:
        counter += 1
        log_message(f"EarlyStopping counter: {counter}/{patience}")

        if counter >= patience:
            log_message("Early stopping triggered!")
            break


# SAVE LAST MODEL
torch.save(model.state_dict(), os.path.join(log_dir, "last_model.pth"))


Epoch 1/10


Epoch 1/10
Train Loss: 3.4414 | Acc: 0.0426
Val   Loss: 3.3829 | Acc: 0.1154
LR: 0.0001
----------------------------------------
Best model saved!

Epoch 2/10


Epoch 2/10
Train Loss: 3.3246 | Acc: 0.0817
Val   Loss: 3.1247 | Acc: 0.1484
LR: 0.0001
----------------------------------------
Best model saved!

Epoch 3/10


Epoch 3/10
Train Loss: 3.1615 | Acc: 0.1221
Val   Loss: 2.9350 | Acc: 0.1264
LR: 0.0001
----------------------------------------
Best model saved!

Epoch 4/10


Epoch 4/10
Train Loss: 2.9425 | Acc: 0.1344
Val   Loss: 2.7046 | Acc: 0.1813
LR: 0.0001
----------------------------------------
Best model saved!

Epoch 5/10


...

Epoch 6/10
Train Loss: 2.5542 | Acc: 0.2128
Val   Loss: 2.2273 | Acc: 0.3626
LR: 0.0001
----------------------------------------
Best model saved!

Epoch 7/10


Epoch 7/10
Train Loss: 2.4751 | Acc: 0.2531
Val   Loss: 2.3435 | Acc: 0.2198
LR: 0.0001
----------------------------------------
EarlyStopping counter: 1/3

Epoch 8/10


Epoch 8/10
Train Loss: 2.2881 | Acc: 0.2822
Val   Loss: 2.0146 | Acc: 0.4231
LR: 0.0001
----------------------------------------
Best model saved!

Epoch 9/10


Epoch 9/10
Train Loss: 2.2297 | Acc: 0.2912
Val   Loss: 1.9718 | Acc: 0.3956
LR: 0.0001
----------------------------------------
Best model saved!

Epoch 10/10


Epoch 10/10
Train Loss: 2.1408 | Acc: 0.3169
Val   Loss: 1.7780 | Acc: 0.4560
LR: 0.0001
----------------------------------------
Best model saved!


In [17]:
model.eval()

test_correct = 0
test_total = 0

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        outputs = model(x)
        preds = torch.argmax(outputs, dim=1)

        test_correct += (preds == y).sum().item()
        test_total += y.size(0)

test_acc = test_correct / test_total

log_message(f"Test Accuracy: {test_acc:.4f}")
print("Test Accuracy:", test_acc)

Test Accuracy: 0.4395
Test Accuracy: 0.43946188340807174
